# MUSANGA V3.1 · NB06 — PlanetScope & embedding spatial information

**Question:** Does fine-scale spatial context add meaningful and transferable information for estimating *Musanga* fraction beyond the established Sentinel-2 and satellite-embedding baseline?

**Stage 1 — exploration, not model fitting:** establish native resolution, available 4-band Planet annual/monthly imagery, annual 9-band NIR texture, 64-band 10 m embeddings and the existing 100 m UAV target cells. Display matched Planet and embedding patches, inspect 30/50/100 m supervision availability and spatial alignment. Compare native PlanetScope canopy texture with spatial variability in embeddings without confusing apparent training fit with generalization.

**Stage 2 — planned ablation, only after patch QA:** model comparison under identical targets and folds:
- **B0:** S2 annual + S2 quarterly + annual satellite embeddings (NB04B baseline).
- **B1:** B0 + handcrafted Planet texture (existing NB05 features).
- **B2:** B0 + multiscale **spatial texture of embeddings** (not merely their 64 central-pixel values).
- **B3:** B0 + Planet spatial CNN features.
- **B4:** B0 + embedding spatial CNN features.
- **B5:** B0 + both Planet and embedding CNN features.

First pilot at 50 m, then 30/100 m. **Same continuous UAV target**, same spatially held-out folds for within-site testing and leave-one-forest-out for geographic transfer. Hold out complete spatial areas/sites *before* patch extraction or learned representation training. Run supervised CNN embedding extraction out-of-fold, and fit all scalers/PCA/normalization from training folds only.

**Constraints:** Annual Planet image is 4.77 m and **may not be cloud-free**. Satellite embeddings are 10 m and already summarize S1/S2 information. Yoko Planet 2021 vs UAV 2020 is a known mismatch. Existing NB04B 100 m samples are used for preview only; no CNN trained in this first stage. No full-ROI 4.77 m imagery is assumed downloaded.


## 00 · Settings — exploratory preview only


In [ ]:
from pathlib import Path
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                          Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL and FINAL.is_dir(),'Musanga Final workspace missing'
ROOT=FINAL/'Outputs/V3_1'
NB05=ROOT/'05_sat_predictor_exploration'
NB04=ROOT/'04_satellite_predictors/input_layers'
NB04B=ROOT/'04B_roi_quarterly_LOSO'
OUT=ROOT/'06_spatial_planet_embeddings'
FIG=OUT/'figures'
for d in (OUT,FIG):d.mkdir(parents=True,exist_ok=True)
SITES=['Baego','MOS','Yangambi','Yoko']
PLANET_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EMBEDDING_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
PREVIEW_PER_SITE=3
PREVIEW_PATCH_PIXELS=48      # ~229m Planet context for each selected 100m cell
EMB_PATCH_PIXELS=23          # ~230m at 10m embedding resolution
RANDOM_SEED=20261024
print('Workspace:',FINAL)


## 01 · Validate source products and review temporal/resolution limitations

Checks actual files, expected 19-band annual Planet, 9-band NIR texture and the existing 64-band annual embeddings. Annual Planet composition bands are:
1–4 screened mean B/G/R/N; 5–8 screened median; 9–12 raw mean; **13–16 raw median**; 17–19 raw valid count / screened valid count / number screened out.
The screened product is **heuristic only**; use the raw median as the stable first CNN baseline.

NB04B's original 100 m sample cache is *not recreated* or overwritten.


In [ ]:
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from rasterio.enums import Resampling
from pyproj import Transformer
from IPython.display import display

def planet_file(site,kind):
    root=NB05/('derived_features' if kind=='Texture' else 'input_layers')
    folder=f'Musanga_V31_NB05_UAV_ROI_{kind}'
    tag=f'NB05_UAV_ROI_{site}_{PLANET_YEAR[site]}_'
    pattern=tag+('Annual_19band*.tif' if kind=='Annual' else 'Texture_9band*.tif')
    matches=sorted((root/folder).glob(pattern))
    if len(matches)!=1:
        raise FileNotFoundError(f'{site} {kind}: expected one file, found {len(matches)}: {matches}')
    return matches[0]

def embed_file(site):
    return NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'

AUDIT=[]
for site in SITES:
    for family,filepath,expected in (
        ('PLANET_ANNUAL',planet_file(site,'Annual'),19),
        ('PLANET_TEXTURE',planet_file(site,'Texture'),9),
        ('EMB_ANNUAL',embed_file(site),64)):
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as src:
            assert src.count==expected,(site,family,'expected',expected,'found',src.count)
            assert src.crs and src.width>0 and src.height>0
            AUDIT.append(dict(site=site,family=family,year=PLANET_YEAR[site] if family!='EMB_ANNUAL' else EMBEDDING_YEAR[site],
                bands=src.count,pixel_size_m=round(abs(src.res[0]),3),
                width=src.width,height=src.height,crs=str(src.crs),
                size_MB=round(filepath.stat().st_size/1e6,2),path=str(filepath)))
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.drop(columns=['path']))
AUDIT.to_csv(OUT/'01_input_audit.csv',index=False)
assert (NB05/'07_final_input_audit/07_site_summary.csv').exists(),'Complete NB05 Section 07 audit first'
print('All 12 native products found: four forests × annual Planet/texture/embeddings.')
print('Note: The 10m embeddings are not 4.77m imagery; do not upscale them to claim new spatial detail.')


## 02 · Select existing 100 m UAV reference cells (identical NB04B targets)

Only select example cells; avoid random point selection from the full Planet image, which would include areas without UAV supervision. Sampling is stratified by low/medium/high Musanga abundance **within each site**, so each forest's previews show more than zero-cover background. Cell centers are computed from NB04B's exact S2-grid transform, not from row/col as if they were Planet raster indices.


In [ ]:
from affine import Affine
SAMPLE_PATH=NB04B/'samples/roi_cells_100m.parquet'
assert SAMPLE_PATH.is_file(),SAMPLE_PATH
DF=pd.read_parquet(SAMPLE_PATH)
assert {'site','row','col','mean_probability'}.issubset(DF.columns)
DF=DF.loc[DF.site.isin(SITES)].copy()
assert len(DF)>0
RNG=np.random.default_rng(RANDOM_SEED)
CENTERS={}
SELECTED=[]
for site in SITES:
    local=DF.loc[DF.site==site].copy()
    assert not local.empty,site
    # Existing 100m NB04B samples use the 10m S2 affine aggregated exactly 10×.
    raw=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    assert raw.is_file(),raw
    with rasterio.open(raw) as ds:
        assert abs(ds.res[0]-10)<.05 and abs(ds.res[1]-10)<.05
        tfm=ds.transform*Affine.scale(10,10)
        crs=ds.crs
    for i in range(PREVIEW_PER_SITE):
        ordered=local.sort_values('mean_probability')
        idx=int(round((i+.5)*(len(ordered)-1)/PREVIEW_PER_SITE))
        sample=ordered.iloc[idx]
        cx,cy=tfm*(float(sample.col)+.5,float(sample.row)+.5)
        SELECTED.append(dict(site=site,group=str(sample.group),mosaic=str(sample.mosaic),
            fraction=float(sample.mean_probability),row=int(sample.row),col=int(sample.col),
            center_x=float(cx),center_y=float(cy),crs=str(crs)))
    CENTERS[site]=dict(transform=tfm,crs=crs)
PREVIEW=pd.DataFrame(SELECTED)
display(PREVIEW[['site','group','fraction','row','col']].round(3))
PREVIEW.to_csv(OUT/'02_preview_cells.csv',index=False)
print('Preview targets are UAV-model estimates at 100 m, NOT independent field labels.')


## 03 · Matched *native* Planet RGB/NIR and embedding-neighborhood previews

For the same UAV-target cell center, read a ~229 m **Planet raw annual median RGB patch** and a ~230 m **10 m satellite-embedding patch**. Embeddings contain 64 latent bands; visualize the first three PCA components as an exploratory false-colour composite (PCA fit only on the displayed patch, **not** for any prediction model). Preserve image pixel scales. This step helps spot spatial shifts, boundary masking, and structural patterns.

The common center is transformed to each raster's CRS if necessary. If a patch crosses a source footprint boundary it is padded with zeros and flagged by nonzero-band masks; no fabricated reflectance values.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
def native_patch(path,cx,cy,center_crs,size,bands):
    with rasterio.open(path) as ds:
        trans=Transformer.from_crs(center_crs,ds.crs,always_xy=True)
        x,y=trans.transform(cx,cy)
        cc,rr=~ds.transform*(x,y)
        start_col=int(np.floor(cc-size/2))
        start_row=int(np.floor(rr-size/2))
        w=Window(start_col,start_row,size,size)
        arr=ds.read(bands,window=w,boundless=True,fill_value=0).astype('float32')
        return arr,ds.crs,ds.res
def rgb_fixed(arr,valid):
    # Input arr is RGB band order; calculate one robust stretch per local patch.
    if np.count_nonzero(valid)<10:return np.ones((*valid.shape,3),dtype='float32')
    p=arr[:,valid]
    lo=np.percentile(p,2,axis=1)
    hi=np.percentile(p,98,axis=1)
    rgb=np.moveaxis(np.clip((arr-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6),0,1),0,-1)
    rgb[~valid]=1
    return rgb
def embed_pca_rgb(cube):
    mask=np.isfinite(cube).all(axis=0)&np.any(cube!=0,axis=0)
    out=np.ones((*mask.shape,3),dtype='float32')
    if mask.sum()<10:return out,mask
    X=cube[:,mask].T
    X=StandardScaler().fit_transform(X)
    vals=PCA(n_components=3).fit_transform(X)
    pct_lo=np.percentile(vals,2,axis=0);pct_hi=np.percentile(vals,98,axis=0)
    out[mask]=np.clip((vals-pct_lo)/(pct_hi-pct_lo+1e-6),0,1)
    return out,mask

fig,axes=plt.subplots(len(PREVIEW),2,figsize=(8,3.0*len(PREVIEW)),layout='constrained')
rows=[]
for i,row in enumerate(PREVIEW.itertuples(index=False)):
    pa=planet_file(row.site,'Annual')
    eb=embed_file(row.site)
    # Planet annual 13-16 = B,G,R,N raw median; select RGB order 15,14,13
    p,_,pres=native_patch(pa,row.center_x,row.center_y,row.crs,PREVIEW_PATCH_PIXELS,[15,14,13])
    e,_,eres=native_patch(eb,row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    pmask=np.all(np.isfinite(p)&(p>0),axis=0)
    eviz,emask=embed_pca_rgb(e)
    axes[i,0].imshow(rgb_fixed(p,pmask));axes[i,1].imshow(eviz)
    axes[i,0].set_title(f'{row.site} | UAV {row.fraction*100:.1f}% | Planet RGB')
    axes[i,1].set_title('10 m embedding neighborhood (local PCA)')
    for ax in axes[i]:ax.axis('off')
    rows.append(dict(site=row.site,fraction=row.fraction,
                     planet_valid_pct=100*pmask.mean(),embedding_valid_pct=100*emask.mean(),
                     planet_pixel_size_m=abs(pres[0]),embedding_pixel_size_m=abs(eres[0])))
plt.savefig(FIG/'03_planet_vs_embedding_native_patches.png',dpi=140,bbox_inches='tight')
plt.show()
PATCH_QA=pd.DataFrame(rows)
display(PATCH_QA.round(2))
PATCH_QA.to_csv(OUT/'03_patch_coverage.csv',index=False)
print('IMPORTANT: local PCA colours are arbitrary across panels/sites and cannot be compared spectrally.')


## 04 · Spatial embeddings vs Planet: first quantitative exploratory features

To assess whether embeddings also contain spatial texture, summarize **within-patch heterogeneity** of their 64 channels. Compare each 100 m target cell's:
- Planet NIR local texture (from NB05's existing annual raster, sampled as local median);
- embedding spatial heterogeneity (median channelwise SD over a ~230 m context), plus embedding central-context mean differences.

**Descriptive only:** 12 deliberately stratified preview cells are too few for useful correlations or model fitting. This diagnostic checks that features are finite and nontrivial. Later we will extract these features across *all* labelled cells, use consistent physical windows (not just fixed pixel counts), and evaluate added value on geographic holdouts.


In [ ]:
from scipy.ndimage import median_filter
QA=[]
for row in PREVIEW.itertuples(index=False):
    e,_,_=native_patch(embed_file(row.site),row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    m=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
    if m.sum()<25:continue
    em=e[:,m]
    vari=np.std(em,axis=1)
    center=e[:,EMB_PATCH_PIXELS//2,EMB_PATCH_PIXELS//2]
    context=np.mean(em,axis=1)
    t,_,_=native_patch(planet_file(row.site,'Texture'),row.center_x,row.center_y,row.crs,
                       PREVIEW_PATCH_PIXELS,[1,2,3,4])
    tm=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)&np.any(t>0,axis=0)
    QA.append(dict(site=row.site,uav_fraction=row.fraction,
        embedding_channel_sd_median=float(np.median(vari)),
        embedding_context_difference=float(np.mean(abs(center-context))),
        planet_NIR_CV_r3_median=float(np.median(t[1,tm])) if tm.any() else np.nan,
        valid_embedding_pct=100*m.mean(),valid_planet_texture_pct=100*tm.mean()))
Q=pd.DataFrame(QA)
display(Q.round(4))
Q.to_csv(OUT/'04_spatial_feature_feasibility.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for site,group in Q.groupby('site'):
    axes[0].scatter(group.uav_fraction*100,group.embedding_channel_sd_median,label=site)
    axes[1].scatter(group.uav_fraction*100,group.planet_NIR_CV_r3_median,label=site)
axes[0].set(xlabel='UAV Musanga fraction (%)',ylabel='Median channel SD in embedding patch',
            title='Embeddings: spatial heterogeneity')
axes[1].set(xlabel='UAV Musanga fraction (%)',ylabel='Planet NIR CV (radius 3)',
            title='Planet: native-scale texture')
axes[1].legend()
plt.savefig(FIG/'04_spatial_heterogeneity_feasibility.png',dpi=150,bbox_inches='tight')
plt.show()
print('This plot has only 12 stratified examples. Do NOT interpret it as prediction skill.')


## 05 · Decision gate and forthcoming modelling experiment

Before CNN model fitting, confirm:
1. Patch-validity percentages and visual alignment of Planet with its target cells; independent UAV orthomosaic/probability overlay will follow when locating the original teacher rasters.
2. Check that Planet **2021** in Yoko is an intentional temporal mismatch against UAV **2020**.
3. Use 50 m targets first and compare 30/100 m; target extraction must use the original **full-resolution UAV probability rasters**, avoiding interpolation of 100 m labels.
4. Train compact models with group/spatial folds: **100 m/50 m patches overlap heavily**, so randomly split cells is not an acceptable test.
5. Compare `B0 → B1/B2 → B3/B4 → B5` using the same site holdout and identical target cells; summarize MAE (pp), RMSE (pp), R², high-cover bias, calibration, and performance separately at MOS. Aggregate CNN output to target cells fairly.

**NB06 currently implements the exploratory entry point (Sections 00–04) and experimental specification; CNN training follows after visual QA.** Avoid building a large training architecture before verifying supervision and patch geometry.
